# RDP Feature Engineering_2

**Author:** Zehra Buse Tüfekçi  
**Date:** 15 April 2026  


## Purpose of This Notebook

This notebook focuses on refining the feature engineering process after identifying data leakage issues in the previous modeling phase.  

In the earlier stage, a model was developed using the first version of feature engineering. Although the model achieved high accuracy, it failed to generalize properly, indicating potential data leakage.  

The goal of this notebook is to eliminate leakage, improve feature quality, and prepare a more reliable dataset for robust model training.


## Scope of Work

In this notebook, the following steps are performed:

- Loading the dataset generated from **Feature Engineering_1**  
- Investigating and identifying features causing data leakage  
- Removing leakage-prone features from the dataset  
- Extracting meaningful information from some of these features and transforming them into new, safer features  
- Refining and reconstructing the feature set  
- Preparing a clean and model-ready dataset  

⚠️ **Note:** This notebook specifically focuses on correcting feature engineering mistakes and preventing data leakage. The modeling process is handled separately to ensure a modular and reliable machine learning pipeline.


## Output

The output of this notebook is a refined dataset that:

- Does not contain data leakage  
- Includes improved and meaningful features  
- Is suitable for reliable model training and evaluation  
- Supports better generalization and real-world performance  

## Importing Libraries & Loading Dataset

In this step, the required libraries for data analysis and visualization are imported.  
Then, the processed dataset obtained from the previous feature engineering stage is loaded into the environment.

The dataset **ride_cancellation_processed_v1.csv** represents the output of the first feature engineering phase and will be used as the base for further improvements.

This step ensures that the dataset is correctly loaded and ready for further analysis and processing.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("ride_cancellation_processed_v1.csv")
df.head()

,Booking Status,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Cancelled Rides by Driver,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Vehicle Type_Auto,...,Payment Method_UPI,Payment Method_Uber Wallet,Year,Month,Day,Day_of_week,Is_weekend,Hour,Minute,target
0,Completed,13.4,25.8,0.0,0.0,627.0,13.58,4.9,4.9,1,...,0,0,2024,8,23,4,0,8,56,0
1,Completed,13.1,28.5,0.0,0.0,416.0,34.02,4.6,5.0,0,...,1,0,2024,10,21,0,0,17,17,0
2,Completed,5.3,19.6,0.0,0.0,737.0,48.21,4.1,4.3,0,...,1,0,2024,9,16,0,0,22,8,0
3,Completed,5.1,18.1,0.0,0.0,316.0,4.85,4.1,4.6,1,...,1,0,2024,2,6,1,0,9,44,0
4,Completed,7.1,20.4,0.0,0.0,640.0,41.24,4.0,4.1,0,...,1,0,2024,6,17,0,0,15,45,0


## Removing Target Leakage Feature

In this step, the **"Booking Status"** column is removed from the dataset.

This feature is excluded because it directly contains outcome-related information, which can lead to **data leakage** if used during model training. Including such a feature would cause the model to learn patterns that would not be available in real-world prediction scenarios.

### Why is this important?
- Prevents the model from "cheating" by using future information  
- Ensures a more realistic and generalizable model  
- Improves the reliability of model evaluation  

In [ ]:
df= df.drop("Booking Status",axis=1)

## Inspecting Dataset Columns

In this step, the column names of the dataset are examined.

Understanding the structure and available features in the dataset is essential before proceeding with further preprocessing and feature engineering.

### Why is this important?
- Provides an overview of all available features  
- Helps identify unnecessary or problematic columns  
- Supports better decision-making for feature selection and transformation  

This step ensures clarity about the dataset’s structure and guides the next stages of the workflow.

In [ ]:
df.columns

Index(['Avg VTAT', 'Avg CTAT', 'Cancelled Rides by Customer',
       'Cancelled Rides by Driver', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Vehicle Type_Auto',
       'Vehicle Type_Bike', 'Vehicle Type_Go Mini', 'Vehicle Type_Go Sedan',
       'Vehicle Type_Premier Sedan', 'Vehicle Type_Uber XL',
       'Vehicle Type_eBike', 'Payment Method_Cash',
       'Payment Method_Credit Card', 'Payment Method_Debit Card',
       'Payment Method_UPI', 'Payment Method_Uber Wallet', 'Year', 'Month',
       'Day', 'Day_of_week', 'Is_weekend', 'Hour', 'Minute', 'target'],
      dtype='object')

## Removing Leakage-Prone Time-Based Features

In this step, the **"Avg VTAT"** and **"Avg CTAT"** columns are removed from the dataset.

High delay values such as **CTAT (Customer Time to Arrival)** and **VTAT (Vehicle Time to Arrival)** showed strong correlation with ride cancellations. However, these values are only known after or during the ride, making them unsuitable for real-time prediction.

### Why is this important?
- These features introduce **data leakage** by including future information  
- They artificially inflate model performance (e.g., high accuracy without real learning)  
- They are not available at the time of prediction in real-world scenarios  

By removing these columns, we ensure that the model only relies on features that are available **before the ride decision is made**, leading to more reliable and generalizable results.

In [ ]:
df= df.drop(["Avg VTAT", "Avg CTAT"],axis=1)

## Creating a Refined Target Variable

In this step, a new target variable called **"new_target"** is created to better represent different cancellation scenarios.

The original **target** variable is extended by incorporating additional information from cancellation-related features.

### Logic Behind the New Target:
- **0 → No cancellation**  
- **1 → Cancelled by Customer**  
- **2 → Cancelled by Driver**  

This transformation allows the problem to shift from a simple binary classification to a more informative multi-class structure.

### Why is this important?
- Captures more detailed behavior in ride cancellations  
- Improves the model’s ability to distinguish between different types of cancellations  
- Provides richer insights for business understanding  

---

## Validating Cancellation Consistency

After creating the new target, a validation step is performed to check the consistency of cancellation-related features.

Specifically, the following check is applied:

- Sum of **"Cancelled Rides by Customer"** and **"Cancelled Rides by Driver"** per row  
- Count the frequency of each possible value  

### Purpose of this validation:
- Ensure there are no conflicting records (e.g., both customer and driver cancellation = 1 at the same time)  
- Detect potential data quality issues  
- Confirm that the dataset follows logical consistency  

This step is critical to ensure that the newly created target variable is reliable and free from hidden data issues.

---
## Removing Redundant Cancellation Features

After creating the new target variable, the original cancellation-related columns (**Cancelled Rides by Customer** and **Cancelled Rides by Driver**) are dropped from the dataset. These features are no longer needed because their information has already been encoded into the new target variable. Keeping them would introduce redundancy and could potentially lead to leakage or biased learning during model training.

In [ ]:
import numpy as np

df["new_target"] = np.where(
    df["target"] == 0, 0,
    np.where(df["Cancelled Rides by Customer"] == 1, 1, 2)
)

In [ ]:
df[["Cancelled Rides by Customer", "Cancelled Rides by Driver"]].sum(axis=1).value_counts()

,count
0.0,93000
1.0,37500


In [ ]:
df = df.drop(['Cancelled Rides by Customer','Cancelled Rides by Driver'],axis=1)

## Removing Original Target Column

In this step, the original **target** column is removed from the dataset after creating the new target variable (**new_target**).

This ensures that the dataset only contains the refined target definition and prevents confusion or redundancy during model training. Keeping the original target alongside the new one could also introduce inconsistencies in the modeling process.

By dropping the original target, we maintain a cleaner and more consistent dataset structure for subsequent machine learning tasks.

In [ ]:
df= df.drop("target",axis=1)

## Feature Engineering from Customer and Driver Ratings

In this step, new features are created based on **`Customer Rating`** and **`Driver Ratings`** in order to better capture behavioral patterns and risk levels in ride interactions.

Instead of using raw continuous rating values, they are transformed into categorical indicators representing different satisfaction levels. This helps the model interpret rating information in a more structured and meaningful way.

### Customer Rating Transformation

**`Customer Rating`** values are divided into three groups:

- Low  
- Medium  
- High  

This transformation allows the model to distinguish between different levels of customer satisfaction rather than relying on subtle numerical differences.

### Driver Rating Transformation

Similarly, **`Driver Ratings`** are also categorized into:

- Low  
- Medium  
- High  

This helps represent driver performance in a clearer and more interpretable way.

### Interaction-Based Feature

In addition to individual transformations, a new feature is created to capture high-risk interactions:

- Low Customer Rating + Low Driver Rating

This feature highlights potentially problematic ride pairs that may be more likely to result in negative outcomes such as cancellations or poor service experiences.

### Final Dataset Preparation

After creating these engineered features, the original columns:

- `Customer Rating`  
- `Driver Ratings`  

are removed from the dataset.

This is done because:

- Raw values are no longer needed after transformation  
- It reduces redundancy  
- It minimizes noise  
- It improves model input quality for the modeling phase  

In [ ]:
df["customer_low_rating"] = (df["Customer Rating"] < 4).astype(int)
df["customer_medium_rating"] = ((df["Customer Rating"] >= 4) & (df["Customer Rating"] < 4.5)).astype(int)
df["customer_high_rating"] = (df["Customer Rating"] >= 4.5).astype(int)

In [ ]:
df["driver_low_rating"] = (df["Driver Ratings"] < 4).astype(int)
df["driver_medium_rating"] = ((df["Driver Ratings"] >= 4) & (df["Driver Ratings"] < 4.5)).astype(int)
df["driver_high_rating"] = (df["Driver Ratings"] >= 4.5).astype(int)

In [ ]:
df["high_risk_pair"] = (
    (df["Driver Ratings"] < 4) &
    (df["Customer Rating"] < 4)
).astype(int)

In [ ]:
df = df.drop(['Driver Ratings','Customer Rating'],axis=1)

## Creating Price Efficiency Feature

In this step, a new feature called **price_per_km** is created by combining booking value and ride distance.

This feature is calculated by dividing the total booking value by the ride distance. A small constant (+1) is added to the denominator to prevent division by zero errors and ensure numerical stability.

This transformation helps represent the **cost efficiency of a ride**, allowing the model to better understand pricing behavior relative to distance.

### Why this feature is important:
- Normalizes booking value with respect to distance  
- Helps capture pricing patterns more effectively than raw values  
- Reduces scale differences between long and short rides  
- Improves model interpretability regarding cost behavior  

In [ ]:
df["price_per_km"] = df["Booking Value"] / (df["Ride Distance"] + 1)

## Creating Time-Based Features

In this step, new features are engineered from the **Hour** variable to capture time-dependent patterns in ride behavior.

Two binary indicators are created to represent different time periods of the day: rush hours and late night hours.

The **rush_hour** feature identifies whether a ride occurs during peak traffic hours (17:00–20:00). This period typically represents high demand, increased congestion, and potentially higher cancellation rates due to delays or availability issues.

The **late_night** feature captures rides occurring between 00:00 and 05:00, a time window associated with lower demand, reduced driver availability, and potentially different user behavior patterns.

### Why these features are important:
- Capture temporal patterns in ride demand and cancellations  
- Help the model learn time-dependent behavioral differences  
- Improve predictive power by incorporating contextual time information  
- Transform a raw time variable into meaningful behavioral signals  

In [ ]:
df["rush_hour"] = df["Hour"].between(17, 20).astype(int)
df["late_night"] = df["Hour"].between(0, 5).astype(int)

## Creating Interaction Feature: Weekend Rush Hours

In this step, a new interaction feature called **weekend_rush** is created by combining weekend information with rush hour periods.

This feature identifies rides that occur specifically during rush hours on weekends, capturing a unique behavioral segment where demand patterns may differ from both regular weekdays and normal weekend hours.

### Why this feature is important:
- Captures interaction effects between time and day type  
- Represents high-demand periods that may have distinct cancellation patterns  
- Helps the model learn more complex behavioral relationships  
- Improves feature expressiveness beyond individual variables  

By combining these two signals, the model can better understand situational demand dynamics in ride behavior.

In [ ]:
df["weekend_rush"] = ((df["Is_weekend"] == 1) & (df["rush_hour"] == 1)).astype(int)

## Removing Irrelevant Time Granularity (Minute Feature)

In this step, the **Minute** column is removed from the dataset because it introduces unnecessary noise without adding meaningful predictive value.

While the **Hour** feature captures overall temporal patterns such as rush hours and late-night behavior, minute-level variations do not significantly change ride cancellation patterns or demand structure in this context.

### Why this feature is removed:
- Minute-level detail does not provide strong predictive signal  
- Increases noise in the dataset rather than useful variation  
- Hour-based features are sufficient to capture time-dependent behavior  
- Helps simplify the feature space and improve model generalization  

By removing this feature, the dataset becomes more stable and focused on meaningful temporal patterns.

In [ ]:
df= df.drop('Minute',axis=1)

## Removing Constant Year Feature

In this step, the **Year** column is removed from the dataset because it does not provide any meaningful variation.

Since the dataset already contains records only from a single year (2024), the **Year** feature has no discriminative power and behaves as a constant value.

### Why this feature is removed:
- Contains no variability (all values are the same year)  
- Does not contribute to model learning or prediction performance  
- Adds unnecessary noise and redundancy to the dataset  
- Helps simplify the feature space and improve model efficiency  

By removing this column, the dataset becomes cleaner and more focused on informative features only.

In [ ]:
df= df.drop('Year',axis=1)

## Removing Low-Value Calendar Granularity (Day Feature)

In this step, the **Day** column is removed from the dataset because it does not provide significant predictive value in the current context.

The dataset already includes higher-level temporal features such as **month information** and **day of the week**, which are more meaningful for capturing seasonal and behavioral patterns. In comparison, the exact day of the month does not contribute additional useful signal for modeling ride behavior.

### Why this feature is removed:
- Day-of-month has weak or no meaningful relationship with the target variable  
- Higher-level time features (month, weekday) already capture relevant patterns  
- Reduces unnecessary noise in the dataset  
- Helps simplify the feature space and improve model generalization  

By removing this column, the dataset focuses only on the most informative temporal signals.

In [ ]:
df= df.drop('Day',axis=1)

## Creating Distance-Price Interaction Feature

In this step, a new binary interaction feature called **distance_high_price** is created using both ride distance and booking value.

This feature identifies rides that are simultaneously above the median in both **distance** and **price**, highlighting high-value and long-distance trips.

### Why this feature is important:
- Captures interaction between distance and pricing behavior  
- Helps identify premium or long-haul rides  
- Provides the model with a non-linear relationship between key variables  
- Improves the ability to distinguish high-value ride segments  

By combining these two conditions, the model can better understand complex patterns that are not visible when each feature is considered independently.

In [ ]:
df["distance_high_price"] = (
    (df["Ride Distance"] > df["Ride Distance"].median()) &
    (df["Booking Value"] > df["Booking Value"].median())
).astype(int)

## Handling Infinite Values in Engineered Feature

In this step, infinite values in the **price_per_km** feature are replaced with missing values (NaN).

These infinite values typically occur when the ride distance is zero or extremely close to zero, leading to division issues during feature engineering.

### Why this step is important:
- Prevents invalid numerical values (∞, -∞) from affecting model training  
- Ensures data consistency and stability in the feature space  
- Allows proper handling of problematic cases in later preprocessing steps (e.g., imputation or removal)  
- Improves overall dataset quality and robustness  

By converting infinite values to NaN, the dataset is made safer and more suitable for downstream modeling.

In [ ]:
df["price_per_km"] = df["price_per_km"].replace([np.inf, -np.inf], np.nan)

## Checking Missing and Infinite Values in the Dataset

In this step, the dataset is checked for **missing (NaN)** and **infinite (inf)** values to ensure data quality and consistency before model training.

### What is being checked:
- Total number of missing values (NaN) in the dataset  
- Total number of infinite values (∞, -∞) in the dataset  

### Why this is important:
- Missing values can negatively affect model training and performance  
- Infinite values can cause computational errors or instability in machine learning algorithms  
- Ensuring clean numerical data is essential for reliable model behavior  

This step acts as a final data quality validation before proceeding to modeling or further preprocessing stages.

In [ ]:
import numpy as np

np.isinf(df).sum()

,0
Booking Value,0
Ride Distance,0
Vehicle Type_Auto,0
Vehicle Type_Bike,0
Vehicle Type_Go Mini,0
Vehicle Type_Go Sedan,0
Vehicle Type_Premier Sedan,0
Vehicle Type_Uber XL,0
Vehicle Type_eBike,0
Payment Method_Cash,0


In [ ]:
print(df.isnull().sum().sum())
print(np.isinf(df).sum().sum())

0
0


## Saving and Downloading the Final Dataset

In this step, the final processed dataset is saved and downloaded for further use.

After completing all preprocessing and feature engineering steps, the dataset is now fully cleaned, leakage-free, and ready for modeling or deployment.

Saving the dataset ensures reproducibility and allows it to be reused in future modeling pipelines without repeating the entire preprocessing workflow.

In [33]:
df.to_csv("final_dataset.csv", index=False)

In [34]:
from google.colab import files
files.download("final_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>